In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModelForVision2Seq, AutoProcessor

DATA_ROOT = Path('/media/chahar/48e17169-ad03-49a2-8ad7-9f071aaf3dde')
FOODSAM_COUNTS_DIR = DATA_ROOT / 'foodsam_item_counts'
NUTRITION_VOLUME_DIR = DATA_ROOT / 'nutrition5k_volume'
FOODSAM_IMAGE_DIR = DATA_ROOT / 'FoodSAM_nutrition5k_outputs'
OUTPUT_CSV = Path('foodqwen_macro_estimates_with_images.csv')
FAILURE_CSV = Path('foodqwen_macro_failures_with_images.csv')

model_id = 'AdaptLLM/food-Qwen2.5-VL-3B-Instruct'
processor = AutoProcessor.from_pretrained(model_id)

if torch.cuda.is_available():
    dtype = torch.float16
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    dtype = torch.float16
    device = torch.device('mps')
else:
    dtype = torch.float32
    device = torch.device('cpu')

model = AutoModelForVision2Seq.from_pretrained(model_id, torch_dtype=dtype)
model.to(device)
model.eval()

print(f"Model loaded on {device} with dtype {dtype}.")

In [ ]:
def collect_dish_ids():
    if not FOODSAM_COUNTS_DIR.exists():
        raise FileNotFoundError(f'Missing FoodSAM counts directory: {FOODSAM_COUNTS_DIR}')
    if not NUTRITION_VOLUME_DIR.exists():
        raise FileNotFoundError(f'Missing Nutrition5k volume directory: {NUTRITION_VOLUME_DIR}')
    if not FOODSAM_IMAGE_DIR.exists():
        raise FileNotFoundError(f'Missing FoodSAM output directory with images: {FOODSAM_IMAGE_DIR}')

    count_ids = {p.stem for p in FOODSAM_COUNTS_DIR.glob('dish_*.json')}
    volume_ids = {
        p.name for p in NUTRITION_VOLUME_DIR.iterdir()
        if p.is_dir() and p.name.startswith('dish_')
    }
    image_ids = {
        p.name for p in FOODSAM_IMAGE_DIR.iterdir()
        if p.is_dir() and (p / 'pred_vis.png').exists() and p.name.startswith('dish_')
    }
    dish_ids = sorted(count_ids & volume_ids & image_ids)
    if not dish_ids:
        raise RuntimeError('No overlapping dish IDs found between counts, volumes, and pred_vis images.')
    return dish_ids


def load_counts(dish_id):
    with open(FOODSAM_COUNTS_DIR / f'{dish_id}.json', 'r') as fp:
        return json.load(fp)


def load_category_volumes(dish_id):
    csv_path = NUTRITION_VOLUME_DIR / dish_id / 'volumes_per_category.csv'
    if not csv_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    numeric_cols = ['volume_m3', 'volume_ml', 'mean_height_cm', 'max_height_cm']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def load_pred_vis_image(dish_id):
    image_path = FOODSAM_IMAGE_DIR / dish_id / 'pred_vis.png'
    if not image_path.exists():
        raise FileNotFoundError(f'Missing pred_vis.png for {dish_id} at {image_path}')
    with Image.open(image_path) as img:
        return img.convert('RGB')


def build_prompt(dish_id, counts_data, volume_df):
    item_lines = []
    for item in counts_data.get('items', []):
        label = item.get('label', 'unknown')
        count = item.get('count', 0)
        mask_ratio_sum = item.get('mask_ratio_sum', 0.0)
        item_lines.append(f"{label}: count={count}, mask_ratio_sum={mask_ratio_sum:.4f}")
    if not item_lines:
        item_lines = ['No detected items']

    if not volume_df.empty:
        volume_lines = [
            f"{row.get('category', 'unknown')}: volume_ml={row.get('volume_ml', float('nan')):.1f}, mean_height_cm={row.get('mean_height_cm', float('nan')):.2f}"
            for _, row in volume_df.iterrows()
        ]
    else:
        volume_lines = ['No category volumes available']

    prompt_sections = [
        f'Dish ID: {dish_id}',
        f"Total food segments: {counts_data.get('total_food_segments', 'unknown')}",
        'Detected food categories with counts and mask ratios:',
        *[f"- {line}" for line in item_lines],
        'Category-level volume estimates (ml):',
        *[f"- {line}" for line in volume_lines],
        'Use the attached FoodSAM visualization image to refine your understanding of the dish.',
        'Estimate dish-level macronutrients (carbs_g, protein_g, fat_g) and calories_kcal.',
        'Provide realistic, non-negative estimates using common food nutrition knowledge.',
        'Respond with a single JSON object only, using this schema:',
        '{"dish_id": "<dish_id>", "carbs_g": <float>, "protein_g": <float>, "fat_g": <float>, "calories_kcal": <float>}.',
        'Use grams (g) for macronutrients and kilocalories (kcal) for energy.'
    ]
    return "\n".join(prompt_sections)


def extract_macros(generated_text, fallback_dish_id):
    if not generated_text:
        return None
    match = re.search(r"\{.*\}", generated_text, re.S)
    if not match:
        return None
    try:
        payload = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

    def to_float(value):
        if value is None:
            return None
        if isinstance(value, (int, float)):
            return float(value)
        if isinstance(value, str):
            cleaned_numbers = re.findall(r"[-+]?[0-9]*\.?[0-9]+", value)
            if cleaned_numbers:
                try:
                    return float(cleaned_numbers[0])
                except ValueError:
                    return None
        return None

    macros = {
        'dish_id': payload.get('dish_id', fallback_dish_id),
        'carbs_g': to_float(payload.get('carbs_g', payload.get('carbs'))),
        'protein_g': to_float(payload.get('protein_g', payload.get('protein'))),
        'fat_g': to_float(payload.get('fat_g', payload.get('fat'))),
        'calories_kcal': to_float(payload.get('calories_kcal', payload.get('calories'))),
    }
    numeric_keys = ['carbs_g', 'protein_g', 'fat_g', 'calories_kcal']
    if any(macros[key] is None for key in numeric_keys):
        return None
    for key in numeric_keys:
        macros[key] = max(0.0, macros[key])
    return macros


def infer_macros(dish_id, counts_data, volume_df, image, max_new_tokens=256):
    prompt = build_prompt(dish_id, counts_data, volume_df)
    user_content = []
    if image is not None:
        user_content.append({'type': 'image', 'image': image})
    user_content.append({'type': 'text', 'text': prompt})

    messages = [
        {
            'role': 'system',
            'content': 'You are a nutrition scientist. Base your estimates on the provided FoodSAM visualization, item counts, and category volumes. Answer with JSON only.'
        },
        {
            'role': 'user',
            'content': user_content,
        },
    ]

    chat_prompt = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    processor_kwargs = dict(return_tensors='pt', padding=True)
    if image is not None:
        inputs = processor(text=[chat_prompt], images=[image], **processor_kwargs)
    else:
        inputs = processor(text=[chat_prompt], **processor_kwargs)

    inputs = {
        key: (value.to(device) if isinstance(value, torch.Tensor) else value)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    prompt_length = inputs['input_ids'].shape[-1]
    response_ids = generated_ids[:, prompt_length:]
    tokenizer = getattr(processor, 'tokenizer', None) or processor
    generated_text = '' if response_ids.shape[1] == 0 else tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0].strip()
    macros = extract_macros(generated_text, dish_id)
    return macros, generated_text


dish_ids = collect_dish_ids()
print(f'Found {len(dish_ids)} dishes with counts, volume data, and pred_vis images.')

MAX_DISHES = None  # Set to an integer to limit the run for testing.
if MAX_DISHES is not None:
    dish_ids = dish_ids[:MAX_DISHES]
    print(f'Processing first {len(dish_ids)} dishes due to MAX_DISHES setting.')

results = []
failures = []

for dish_id in tqdm(dish_ids, desc='Estimating macros with images'):
    counts_data = load_counts(dish_id)
    volume_df = load_category_volumes(dish_id)
    try:
        image = load_pred_vis_image(dish_id)
    except FileNotFoundError as exc:
        failures.append({'dish_id': dish_id, 'raw_response': str(exc)})
        continue

    macros, raw_text = infer_macros(dish_id, counts_data, volume_df, image)
    if macros:
        results.append(macros)
    else:
        failures.append({'dish_id': dish_id, 'raw_response': raw_text})

results_df = pd.DataFrame(results)
if not results_df.empty:
    results_df = results_df[['dish_id', 'carbs_g', 'protein_g', 'fat_g', 'calories_kcal']]
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f'Saved {len(results_df)} dish macro estimates to {OUTPUT_CSV.resolve()}.')
else:
    print('No macro estimates were produced.')

if failures:
    failure_df = pd.DataFrame(failures)
    failure_df.to_csv(FAILURE_CSV, index=False)
    print(f"{len(failures)} dishes failed to parse. Details saved to {FAILURE_CSV.resolve()}.")
else:
    print('All dishes parsed successfully.')